# LeetCode 1334 — Find the City With the Smallest Number of Neighbors at a Threshold Distance bằng Python

Link đề: https://leetcode.com/problems/find-the-city-with-the-smallest-number-of-neighbors-at-a-threshold-distance/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming, Graph Theory, Shortest Path, Dijkstra's Algorithm, Bellman–Ford Algorithm, Floyd–Warshall Algorithm

Skeleton LeetCode:

```python
class Solution:
    def findTheCity(self, n: int, edges: list[list[int]], distanceThreshold: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hoá thành **đồ thị vô hướng có trọng số không âm** và hiểu "đến được trong ngưỡng" = **khoảng cách ngắn nhất** `≤ distanceThreshold`.
2. Nắm ba thuật toán đường đi ngắn nhất kinh điển: **Floyd–Warshall** (mọi cặp, `O(n³)`), **Dijkstra** từ mọi đỉnh (heap), **Bellman–Ford** từ mọi đỉnh.
3. Hiểu vì sao Floyd–Warshall là **quy hoạch động** và vì sao vòng `k` phải nằm **ngoài cùng**.
4. Xử lý đúng **quy tắc hoà**: nhiều thành phố cùng số láng giềng ít nhất ⇒ chọn **id lớn nhất**.
5. Tránh lỗi: chọn id nhỏ nhất khi hoà, đặt sai thứ tự vòng lặp Floyd, quên cạnh hai chiều, chọn `INF` quá nhỏ.

## 2. Đề bài & ví dụ

Có `n` thành phố `0..n−1`. Mảng `edges[i] = [from, to, weight]` là một con đường **hai chiều** có độ dài `weight`. Cho `distanceThreshold`.

Với mỗi thành phố, đếm số thành phố **khác** mà từ nó đến được bằng một đường đi có tổng độ dài **không vượt quá** `distanceThreshold`. Trả về thành phố có **số đếm nhỏ nhất**; nếu có nhiều, trả về thành phố có **chỉ số lớn nhất**.

**Ví dụ 1:** `n = 4, edges = [[0,1,3],[1,2,1],[1,3,4],[2,3,1]], distanceThreshold = 4` → `3`
(Thành phố 0 → {1, 2}; 1 → {0, 2, 3}; 2 → {0, 1, 3}; 3 → {1, 2}. Thành phố 0 và 3 cùng có 2, chọn 3.)

**Ví dụ 2:** `n = 5, edges = [[0,1,2],[0,4,8],[1,2,3],[1,4,2],[2,3,1],[3,4,1]], distanceThreshold = 2` → `0`
(0 → {1}; 1 → {0, 4}; 2 → {3, 4}; 3 → {2, 4}; 4 → {1, 2, 3}.)

**Ràng buộc:** `2 <= n <= 100`; `1 <= edges.length <= n(n−1)/2`; `0 <= from < to < n`; `1 <= weight, distanceThreshold <= 10^4`; mọi cặp `(from, to)` phân biệt.

In [1]:
import heapq
import random
import time
from typing import List

examples = [
    (4, [[0, 1, 3], [1, 2, 1], [1, 3, 4], [2, 3, 1]], 4, 3),
    (5, [[0, 1, 2], [0, 4, 8], [1, 2, 3], [1, 4, 2], [2, 3, 1], [3, 4, 1]], 2, 0),
]
for n, edges, t, want in examples:
    print(f"n={n}, threshold={t}, edges={edges} -> mong đợi {want}")

n=4, threshold=4, edges=[[0, 1, 3], [1, 2, 1], [1, 3, 4], [2, 3, 1]] -> mong đợi 3
n=5, threshold=2, edges=[[0, 1, 2], [0, 4, 8], [1, 2, 3], [1, 4, 2], [2, 3, 1], [3, 4, 1]] -> mong đợi 0


## 3. Kiến thức nền

### 3.1 Danh sách kề có trọng số

`graph[u] = [(v, w), ...]`. Đường **hai chiều** ⇒ thêm cả `(v, w)` vào `graph[u]` và `(u, w)` vào `graph[v]`.

In [2]:
def build_graph(n, edges):
    graph = [[] for _ in range(n)]
    for u, v, w in edges:
        graph[u].append((v, w))
        graph[v].append((u, w))
    return graph

g = build_graph(*examples[0][:2])
for u, adj in enumerate(g):
    print(u, "->", adj)

0 -> [(1, 3)]
1 -> [(0, 3), (2, 1), (3, 4)]
2 -> [(1, 1), (3, 1)]
3 -> [(1, 4), (2, 1)]


### 3.2 Ma trận khoảng cách và vô cực

`dist[i][j]` = độ dài đường ngắn nhất đã biết từ `i` đến `j`. Khởi tạo: `0` trên đường chéo, `w` nếu có cạnh, `INF` nếu chưa biết. Dùng `float('inf')` (cộng vẫn là `inf`, so sánh đúng) hoặc một số **đủ lớn** (lớn hơn mọi đường đi có thể, ví dụ `10^9`).

In [3]:
INF = float("inf")
n = 4
dist = [[INF] * n for _ in range(n)]
for i in range(n):
    dist[i][i] = 0
for u, v, w in examples[0][1]:
    dist[u][v] = dist[v][u] = w
for row in dist:
    print(row)
print(INF + 5 > 10**9, min(INF, 7))

[0, 3, inf, inf]
[3, 0, 1, 4]
[inf, 1, 0, 1]
[inf, 4, 1, 0]
True 7


### 3.3 Min-heap với `heapq`

`heapq.heappush(h, (d, u))`, `heapq.heappop(h)` trả phần tử có `d` nhỏ nhất (so sánh tuple theo phần tử đầu). Đây là "hàng đợi ưu tiên" cho Dijkstra.

In [4]:
h = []
for item in [(5, "C"), (1, "A"), (3, "B")]:
    heapq.heappush(h, item)
print([heapq.heappop(h) for _ in range(3)])

[(1, 'A'), (3, 'B'), (5, 'C')]


### 3.4 Phép "nới lỏng" (relaxation)

Trái tim của mọi thuật toán đường đi ngắn nhất: nếu đi qua `u` rồi theo cạnh `(u, v, w)` cho đường ngắn hơn ⇒ cập nhật:

```python
if dist[u] + w < dist[v]:
    dist[v] = dist[u] + w
```

In [5]:
d = {"A": 0, "B": 10, "C": INF}
for u, v, w in [("A", "B", 3), ("B", "C", 4), ("A", "C", 9)]:
    if d[u] + w < d[v]:
        print(f"nới lỏng {u}->{v}: {d[v]} -> {d[u] + w}")
        d[v] = d[u] + w
print(d)

nới lỏng A->B: 10 -> 3
nới lỏng B->C: inf -> 7
{'A': 0, 'B': 3, 'C': 7}


### 3.5 Chọn phần tử nhỏ nhất với quy tắc hoà

"Số đếm nhỏ nhất, hoà thì id lớn nhất" ⇒ duyệt `i` **tăng dần** và cập nhật khi `cnt <= best_cnt` (dấu `<=` để id sau ghi đè). Hoặc gọn: `min(range(n), key=lambda i: (cnt[i], -i))`.

In [6]:
cnt = [2, 3, 3, 2]
best, best_cnt = -1, float("inf")
for i in range(len(cnt)):
    if cnt[i] <= best_cnt:
        best, best_cnt = i, cnt[i]
print(best, min(range(len(cnt)), key=lambda i: (cnt[i], -i)))

3 3


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** (threshold `4`). Vẽ đồ thị: `0 –3– 1 –1– 2 –1– 3` và thêm cạnh `1 –4– 3`.

Khoảng cách ngắn nhất (tính bằng mắt):

| từ \ đến | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| 0 | 0 | 3 | 4 (0→1→2) | 5 (0→1→2→3) |
| 1 | 3 | 0 | 1 | 2 (1→2→3, ngắn hơn cạnh trực tiếp 4) |
| 2 | 4 | 1 | 0 | 1 |
| 3 | 5 | 2 | 1 | 0 |

Đếm các ô `≤ 4` (trừ chính nó): 0 → 2, 1 → 3, 2 → 3, 3 → 2. Nhỏ nhất là 2 ở thành phố 0 và 3 ⇒ chọn **3** ✔.

Quan sát: cạnh trực tiếp `1–3` dài 4 **không** phải đường ngắn nhất ⇒ phải tính khoảng cách ngắn nhất thật sự, không chỉ nhìn cạnh kề.

In [7]:
paths = {(0, 2): [0, 1, 2], (0, 3): [0, 1, 2, 3], (1, 3): [1, 2, 3]}
w = {}
for u, v, ww in examples[0][1]:
    w[(u, v)] = w[(v, u)] = ww
for (a, b), p in paths.items():
    print(f"{a}->{b} qua {p}: dài {sum(w[(p[i], p[i + 1])] for i in range(len(p) - 1))}")

0->2 qua [0, 1, 2]: dài 4
0->3 qua [0, 1, 2, 3]: dài 5
1->3 qua [1, 2, 3]: dài 2


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** từ mỗi thành phố, **DFS liệt kê mọi đường đi đơn** (không lặp đỉnh), cắt nhánh khi tổng độ dài vượt ngưỡng. Mọi đỉnh chạm được bởi một đường như vậy là "đến được". Vì trọng số dương, đường ngắn nhất luôn là đường đơn ⇒ cách này đúng.

In [8]:
def find_the_city_brute(n, edges, distanceThreshold):
    graph = build_graph(n, edges)

    def reachable_from(src):
        seen_ok = set()
        on_path = [False] * n

        def dfs(u, d):
            seen_ok.add(u)
            on_path[u] = True
            for v, w in graph[u]:
                if not on_path[v] and d + w <= distanceThreshold:
                    dfs(v, d + w)
            on_path[u] = False

        dfs(src, 0)
        return len(seen_ok) - 1          # trừ chính nó

    best, best_cnt = -1, n + 1
    for i in range(n):
        c = reachable_from(i)
        if c <= best_cnt:
            best, best_cnt = i, c
    return best

for n, edges, t, want in examples:
    print(find_the_city_brute(n, edges, t), "| mong đợi", want)

3 | mong đợi 3
0 | mong đợi 0


**Độ phức tạp:** số đường đi đơn có thể **bùng nổ theo cấp số mũ** (đồ thị đầy đủ có tới `(n−1)!` đường). Chỉ dùng được với `n ≤ 8` hoặc ngưỡng nhỏ. Bộ nhớ `O(n)` cho đệ quy.

**Nút thắt:** cùng một đỉnh bị thăm lại qua rất nhiều đường khác nhau. Điều ta cần chỉ là **độ dài ngắn nhất** đến mỗi đỉnh — đúng việc của các thuật toán đường đi ngắn nhất.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** "đến được trong ngưỡng" ⇔ `dist(i, j) <= distanceThreshold` với `dist` là khoảng cách **ngắn nhất**. Vậy chỉ cần ma trận khoảng cách mọi cặp, rồi đếm.

**Insight 2 — Floyd–Warshall là DP:** gọi `D_k[i][j]` = đường ngắn nhất từ `i` đến `j` chỉ được đi **qua các đỉnh trung gian thuộc `{0..k−1}`**. Khi cho phép thêm đỉnh `k`:

`D_{k+1}[i][j] = min(D_k[i][j], D_k[i][k] + D_k[k][j])`

(hoặc không đi qua `k`, hoặc đi qua `k` đúng một lần). Có thể cập nhật **tại chỗ** trên một ma trận. Vòng `k` **phải ở ngoài cùng**.

**Insight 3 — Dijkstra:** trọng số không âm ⇒ từ mỗi nguồn, Dijkstra cho khoảng cách đến mọi đỉnh. Có thể **dừng sớm** khi khoảng cách lấy ra khỏi heap vượt ngưỡng.

**Quy tắc chọn:** duyệt `i = 0..n−1`, nếu `cnt[i] <= best_cnt` thì cập nhật (hoà ⇒ id lớn hơn thắng).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Floyd–Warshall

**Từng đoạn:**
1. Khởi tạo `dist` (`0` đường chéo, `w` cho cạnh, `INF` còn lại) — nhớ **hai chiều**.
2. Ba vòng lặp `k` (ngoài), `i`, `j`: `dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])`.
3. Đếm cho mỗi `i` số `j ≠ i` có `dist[i][j] <= threshold`, chọn theo quy tắc hoà.

Mẹo tăng tốc trong Python: nếu `dist[i][k]` là `INF` hoặc đã `> threshold` thì đi qua `k` không giúp ích cho việc đếm ⇒ bỏ qua cả hàng `i`. Ở bản demo ta viết thuần, bản nộp có mẹo này.

In [9]:
def floyd(n, edges):
    INF = float("inf")
    dist = [[INF] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
    for u, v, w in edges:
        dist[u][v] = min(dist[u][v], w)
        dist[v][u] = min(dist[v][u], w)
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    return dist

def pick_city(n, dist, t):
    best, best_cnt = -1, n + 1
    for i in range(n):
        c = sum(1 for j in range(n) if j != i and dist[i][j] <= t)
        if c <= best_cnt:
            best, best_cnt = i, c
    return best

def find_the_city_floyd(n, edges, distanceThreshold):
    return pick_city(n, floyd(n, edges), distanceThreshold)

for n, edges, t, want in examples:
    print(find_the_city_floyd(n, edges, t), "| mong đợi", want)

3 | mong đợi 3
0 | mong đợi 0


Code nộp LeetCode (Cách 2):

In [10]:
class SolutionFloyd:
    def findTheCity(self, n: int, edges: list[list[int]], distanceThreshold: int) -> int:
        INF = 10**9
        dist = [[INF] * n for _ in range(n)]
        for i in range(n):
            dist[i][i] = 0
        for u, v, w in edges:
            dist[u][v] = dist[v][u] = w
        for k in range(n):
            dk = dist[k]
            for i in range(n):
                dik = dist[i][k]
                if dik >= distanceThreshold:      # đi qua k không tạo đường <= ngưỡng mới
                    continue
                di = dist[i]
                for j in range(n):
                    if dik + dk[j] < di[j]:
                        di[j] = dik + dk[j]
        best, best_cnt = -1, n + 1
        for i in range(n):
            c = sum(1 for d in dist[i] if d <= distanceThreshold) - 1
            if c <= best_cnt:
                best, best_cnt = i, c
        return best

print([SolutionFloyd().findTheCity(*e[:3]) for e in examples])

[3, 0]


*Vì sao mẹo `dik >= threshold` an toàn?* Nó có thể khiến một số `dist[i][j]` lớn hơn ngưỡng không được cập nhật về giá trị thật, nhưng mọi đường qua `k` khi đó dài `≥ dik + dk[j] > threshold` (vì `dk[j] ≥ 1` với `j ≠ k`), nên **không bao giờ** làm một cặp trở thành "đến được". Mọi cặp có khoảng cách thật `≤ threshold` vẫn được tính đúng (đường ngắn nhất của chúng chỉ đi qua các đoạn con đều `< threshold`). Ta kiểm chứng bằng test ngẫu nhiên ở mục 12.

**Độ phức tạp:** `O(n³)` thời gian (`10^6` với `n = 100`), `O(n²)` bộ nhớ.

### 7.2 Cách 3 — Dijkstra từ mọi đỉnh (dừng sớm theo ngưỡng)

**Từng đoạn:**
1. Danh sách kề hai chiều.
2. Với mỗi nguồn `s`: `dist = {s: 0}`, heap `[(0, s)]`. Lấy `(d, u)` nhỏ nhất; nếu `d > dist[u]` (bản ghi cũ) bỏ qua; nới lỏng các cạnh nếu `d + w <= threshold` (không cần đẩy những đỉnh vượt ngưỡng).
3. Số láng giềng = `len(dist) − 1`.

In [11]:
def count_within_dijkstra(graph, s, t):
    dist = {s: 0}
    heap = [(0, s)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        for v, w in graph[u]:
            nd = d + w
            if nd <= t and nd < dist.get(v, float("inf")):
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return len(dist) - 1

def find_the_city_dijkstra(n, edges, distanceThreshold):
    graph = build_graph(n, edges)
    best, best_cnt = -1, n + 1
    for s in range(n):
        c = count_within_dijkstra(graph, s, distanceThreshold)
        if c <= best_cnt:
            best, best_cnt = s, c
    return best

for n, edges, t, want in examples:
    print(find_the_city_dijkstra(n, edges, t), "| mong đợi", want)

3 | mong đợi 3
0 | mong đợi 0


Code nộp LeetCode (Cách 3):

In [12]:
class SolutionDijkstra:
    def findTheCity(self, n: int, edges: list[list[int]], distanceThreshold: int) -> int:
        graph = [[] for _ in range(n)]
        for u, v, w in edges:
            graph[u].append((v, w))
            graph[v].append((u, w))
        best, best_cnt = -1, n + 1
        for s in range(n):
            dist = [float("inf")] * n
            dist[s] = 0
            heap = [(0, s)]
            while heap:
                d, u = heapq.heappop(heap)
                if d > dist[u]:
                    continue
                for v, w in graph[u]:
                    nd = d + w
                    if nd <= distanceThreshold and nd < dist[v]:
                        dist[v] = nd
                        heapq.heappush(heap, (nd, v))
            c = sum(1 for d in dist if d <= distanceThreshold) - 1
            if c <= best_cnt:
                best, best_cnt = s, c
        return best

print([SolutionDijkstra().findTheCity(*e[:3]) for e in examples])

[3, 0]


**Độ phức tạp:** mỗi Dijkstra `O((n + E) log n)` ⇒ tổng `O(n·(n + E)·log n)`; với đồ thị thưa nhanh hơn Floyd rõ rệt, với đồ thị đầy đủ (`E ≈ n²/2`) xấp xỉ `O(n³ log n)` nhưng nhờ cắt theo ngưỡng thường nhanh hơn nhiều. Bộ nhớ `O(n + E)`.

### 7.3 Cách 4 — Bellman–Ford từ mọi đỉnh

**Ý tưởng:** từ nguồn `s`, lặp tối đa `n − 1` lượt, mỗi lượt nới lỏng **mọi cạnh** (cả hai chiều). Dừng sớm nếu một lượt không cập nhật gì. Không cần heap, xử lý được cả trọng số âm (bài này không có) — chậm hơn nhưng là thuật toán nền tảng.

In [13]:
def bellman_ford(n, edges, s):
    INF = float("inf")
    dist = [INF] * n
    dist[s] = 0
    for _ in range(n - 1):
        changed = False
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                changed = True
            if dist[v] + w < dist[u]:
                dist[u] = dist[v] + w
                changed = True
        if not changed:
            break
    return dist

def find_the_city_bellman(n, edges, distanceThreshold):
    best, best_cnt = -1, n + 1
    for s in range(n):
        dist = bellman_ford(n, edges, s)
        c = sum(1 for d in dist if d <= distanceThreshold) - 1
        if c <= best_cnt:
            best, best_cnt = s, c
    return best

for n, edges, t, want in examples:
    print(find_the_city_bellman(n, edges, t), "| mong đợi", want)

3 | mong đợi 3
0 | mong đợi 0


Code nộp LeetCode (Cách 4 — đúng nhưng có thể chậm với đồ thị dày):

In [14]:
class SolutionBellman:
    def findTheCity(self, n: int, edges: list[list[int]], distanceThreshold: int) -> int:
        best, best_cnt = -1, n + 1
        INF = float("inf")
        for s in range(n):
            dist = [INF] * n
            dist[s] = 0
            for _ in range(n - 1):
                changed = False
                for u, v, w in edges:
                    if dist[u] + w < dist[v]:
                        dist[v] = dist[u] + w
                        changed = True
                    if dist[v] + w < dist[u]:
                        dist[u] = dist[v] + w
                        changed = True
                if not changed:
                    break
            c = sum(1 for d in dist if d <= distanceThreshold) - 1
            if c <= best_cnt:
                best, best_cnt = s, c
        return best

print([SolutionBellman().findTheCity(*e[:3]) for e in examples])

[3, 0]


**Độ phức tạp:** mỗi nguồn `O(n·E)` xấu nhất ⇒ tổng `O(n²·E)` (đồ thị đầy đủ `n = 100`: ~`5·10^7` — chậm trong Python; thực tế nhờ dừng sớm số lượt thường nhỏ). Bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace Floyd–Warshall trên **Ví dụ 1**: in ma trận sau mỗi giá trị `k` và các ô thay đổi.

| k | ô thay đổi (lý do) |
|---|---|
| 0 | không có (0 chỉ nối với 1) |
| 1 | `0↔2 = 3+1 = 4`, `0↔3 = 3+4 = 7` |
| 2 | `0↔3 = 4+1 = 5`, `1↔3 = 1+1 = 2` |
| 3 | không có |

In [15]:
def trace_floyd(n, edges, t):
    INF = float("inf")
    dist = [[INF] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
    for u, v, w in edges:
        dist[u][v] = dist[v][u] = w
    fmt = lambda x: "  ∞" if x == INF else f"{x:3}"
    print("khởi tạo:")
    for row in dist:
        print("   ", " ".join(fmt(x) for x in row))
    for k in range(n):
        changes = []
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    changes.append(f"({i},{j}): {fmt(dist[i][j]).strip()} -> {dist[i][k] + dist[k][j]}")
                    dist[i][j] = dist[i][k] + dist[k][j]
        print(f"k={k}: {changes if changes else 'không đổi'}")
    print("ma trận cuối:")
    for i, row in enumerate(dist):
        cnt = sum(1 for j in range(n) if j != i and row[j] <= t)
        print("   ", " ".join(fmt(x) for x in row), f"| số láng giềng ≤ {t}: {cnt}")
    return dist

_ = trace_floyd(*examples[0][:3])

khởi tạo:
      0   3   ∞   ∞
      3   0   1   4
      ∞   1   0   1
      ∞   4   1   0
k=0: không đổi
k=1: ['(0,2): ∞ -> 4', '(0,3): ∞ -> 7', '(2,0): ∞ -> 4', '(3,0): ∞ -> 7']
k=2: ['(0,3): 7 -> 5', '(1,3): 4 -> 2', '(3,0): 7 -> 5', '(3,1): 4 -> 2']
k=3: không đổi
ma trận cuối:
      0   3   4   5 | số láng giềng ≤ 4: 2
      3   0   1   2 | số láng giềng ≤ 4: 3
      4   1   0   1 | số láng giềng ≤ 4: 3
      5   2   1   0 | số láng giềng ≤ 4: 2


Trace Dijkstra từ thành phố 4 trong **Ví dụ 2** (threshold 2):

In [16]:
def trace_dijkstra(n, edges, s, t):
    graph = build_graph(n, edges)
    dist = {s: 0}
    heap = [(0, s)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            print(f"  pop ({d},{u}) cũ -> bỏ qua")
            continue
        pushed = []
        for v, w in graph[u]:
            nd = d + w
            if nd <= t and nd < dist.get(v, float("inf")):
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
                pushed.append((nd, v))
        print(f"  pop ({d},{u}) -> đẩy {pushed}, heap={sorted(heap)}")
    print(f"từ {s}: khoảng cách ≤ {t} = {dict(sorted(dist.items()))}, số láng giềng = {len(dist) - 1}")

trace_dijkstra(*examples[1][:2], 4, examples[1][2])

  pop (0,4) -> đẩy [(2, 1), (1, 3)], heap=[(1, 3), (2, 1)]
  pop (1,3) -> đẩy [(2, 2)], heap=[(2, 1), (2, 2)]
  pop (2,1) -> đẩy [], heap=[(2, 2)]
  pop (2,2) -> đẩy [], heap=[]
từ 4: khoảng cách ≤ 2 = {1: 2, 2: 2, 3: 1, 4: 0}, số láng giềng = 3


## 9. Code nộp LeetCode

Bản chọn nộp: Floyd–Warshall (ngắn, `n ≤ 100` nên `O(n³)` thoải mái), có mẹo bỏ qua hàng.

In [17]:
class Solution:
    def findTheCity(self, n: int, edges: list[list[int]], distanceThreshold: int) -> int:
        INF = 10**9
        dist = [[INF] * n for _ in range(n)]
        for i in range(n):
            dist[i][i] = 0
        for u, v, w in edges:                         # đường hai chiều
            dist[u][v] = dist[v][u] = w
        for k in range(n):                            # k phải ở vòng ngoài cùng
            dk = dist[k]
            for i in range(n):
                dik = dist[i][k]
                if dik >= INF:
                    continue
                di = dist[i]
                for j in range(n):
                    if dik + dk[j] < di[j]:
                        di[j] = dik + dk[j]
        best, best_cnt = -1, n + 1
        for i in range(n):
            cnt = sum(1 for j in range(n) if j != i and dist[i][j] <= distanceThreshold)
            if cnt <= best_cnt:                       # '<=' : hoà thì id lớn hơn thắng
                best, best_cnt = i, cnt
        return best

for n, edges, t, want in examples:
    got = Solution().findTheCity(n, edges, t)
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

3 | mong đợi 3 | OK
0 | mong đợi 0 | OK


## 10. Vì sao thuật toán đúng

**Floyd–Warshall (quy nạp theo `k`):** Bất biến: sau khi xử lý xong `k`, `dist[i][j]` = độ dài đường ngắn nhất từ `i` đến `j` chỉ dùng đỉnh trung gian trong `{0..k}`. Cơ sở (trước vòng lặp): chỉ có cạnh trực tiếp. Bước `k`: một đường ngắn nhất dùng trung gian trong `{0..k}` hoặc không đi qua `k` (giá trị cũ), hoặc đi qua `k` đúng một lần (trọng số dương ⇒ không lặp đỉnh), khi đó tách thành `i ⇝ k` và `k ⇝ j`, mỗi đoạn chỉ dùng trung gian trong `{0..k−1}` ⇒ `dist[i][k] + dist[k][j]`. Cập nhật tại chỗ an toàn vì `dist[i][k]` và `dist[k][j]` không đổi trong vòng `k` (`dist[k][k] = 0`). Sau `k = n−1`, `dist` là khoảng cách ngắn nhất thật.

**Dijkstra:** với trọng số không âm, mỗi lần lấy ra đỉnh có `d` nhỏ nhất, `d` đã là khoảng cách cuối cùng (mọi đường khác phải đi qua một đỉnh còn trong heap với khoảng cách `≥ d`). Cắt các giá trị `> threshold` không ảnh hưởng đỉnh trong ngưỡng: đường ngắn nhất tới một đỉnh trong ngưỡng chỉ gồm các tiền tố cũng trong ngưỡng.

**Chọn thành phố:** duyệt tăng dần với `<=` ⇒ giữ thành phố cuối cùng (id lớn nhất) trong số những thành phố đạt số đếm nhỏ nhất.

## 11. Phân tích Time / Space

Đặt `n` = số thành phố (`≤ 100`), `E` = số cạnh (`≤ n(n−1)/2 ≈ 4950`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. DFS mọi đường đơn | mũ (tới `O(n!)`) | `O(n)` | chỉ `n` rất nhỏ |
| 2. Floyd–Warshall | `O(n³)` | `O(n²)` | đơn giản, hợp `n ≤ 100` |
| 3. Dijkstra × n | `O(n·(n + E)·log n)` | `O(n + E)` | tốt với đồ thị thưa, dừng sớm theo ngưỡng |
| 4. Bellman–Ford × n | `O(n²·E)` | `O(n + E)` | chậm nhất, cho phép trọng số âm |

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
solvers = {
    "brute": find_the_city_brute,
    "floyd": find_the_city_floyd,
    "dijkstra": find_the_city_dijkstra,
    "bellman": find_the_city_bellman,
    "submit": lambda *a: Solution().findTheCity(*a),
    "submit_floyd": lambda *a: SolutionFloyd().findTheCity(*a),
    "submit_dijk": lambda *a: SolutionDijkstra().findTheCity(*a),
    "submit_bell": lambda *a: SolutionBellman().findTheCity(*a),
}
cases = examples + [
    (2, [[0, 1, 5]], 4, 1),                         # không ai đến được ai -> hoà 0, chọn 1
    (2, [[0, 1, 5]], 5, 1),                         # đúng bằng ngưỡng -> đều 1, chọn 1
    (3, [[0, 1, 1]], 10, 2),                        # đỉnh 2 cô lập: 0 láng giềng
    (4, [[0, 1, 1], [1, 2, 1], [2, 3, 1]], 1, 3),   # đường thẳng: đầu mút có 1, chọn 3
    (4, [[0, 1, 1], [1, 2, 1], [2, 3, 1]], 2, 3),   # 0:2, 1:3, 2:3, 3:2 -> 3
    (3, [[0, 1, 10], [1, 2, 1], [0, 2, 1]], 2, 2),  # đường gián tiếp 0-2-1 dài 2 ngắn hơn cạnh 10
]
for n, edges, t, want in cases:
    for name, f in solvers.items():
        assert f(n, edges, t) == want, (name, n, edges, t, f(n, edges, t), want)
print("Tất cả", len(cases), "case qua cho", len(solvers), "lời giải")

Tất cả 8 case qua cho 8 lời giải


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [19]:
def random_graph(rng, n, p, wmax):
    edges = []
    for u in range(n):
        for v in range(u + 1, n):
            if rng.random() < p:
                edges.append([u, v, rng.randint(1, wmax)])
    if not edges:
        u, v = sorted(rng.sample(range(n), 2))
        edges.append([u, v, rng.randint(1, wmax)])
    rng.shuffle(edges)
    return edges

rng = random.Random(1334)
for _ in range(400):
    n = rng.randint(2, 7)
    edges = random_graph(rng, n, rng.choice([0.3, 0.6, 1.0]), rng.choice([3, 10]))
    t = rng.randint(1, 15)
    want = find_the_city_brute(n, edges, t)
    for name, f in solvers.items():
        assert f(n, edges, t) == want, (name, n, edges, t)
print("400 đồ thị ngẫu nhiên nhỏ: mọi cách khớp brute force")

for _ in range(60):
    n = rng.randint(8, 40)
    edges = random_graph(rng, n, rng.choice([0.05, 0.2, 0.5]), 100)
    t = rng.randint(1, 300)
    res = {name: f(n, edges, t) for name, f in solvers.items() if name != "brute"}
    assert len(set(res.values())) == 1, (n, t, res)
print("60 đồ thị vừa: Floyd / Dijkstra / Bellman–Ford trùng nhau")

400 đồ thị ngẫu nhiên nhỏ: mọi cách khớp brute force


60 đồ thị vừa: Floyd / Dijkstra / Bellman–Ford trùng nhau


### 12.3 Kích thước lớn nhất: `n = 100`, đồ thị đầy đủ

In [20]:
rng = random.Random(100)
n = 100
full = [[u, v, rng.randint(1, 10**4)] for u in range(n) for v in range(u + 1, n)]
for t in (10**4, 3000):
    for name in ["submit", "submit_floyd", "submit_dijk"]:
        t0 = time.time()
        r = solvers[name](n, full, t)
        print(f"threshold={t:5} {name:13}: {r}  ({time.time() - t0:.3f}s)")

threshold=10000 submit       : 99  (0.071s)
threshold=10000 submit_floyd : 99  (0.068s)


threshold=10000 submit_dijk  : 99  (0.107s)


threshold= 3000 submit       : 99  (0.059s)
threshold= 3000 submit_floyd : 99  (0.039s)
threshold= 3000 submit_dijk  : 99  (0.052s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Hoà thì chọn id nhỏ nhất (dùng `<`)

In [21]:
def wrong_tie(n, edges, distanceThreshold):
    dist = floyd(n, edges)
    best, best_cnt = -1, n + 1
    for i in range(n):
        c = sum(1 for j in range(n) if j != i and dist[i][j] <= distanceThreshold)
        if c < best_cnt:                    # SAI: giữ id nhỏ nhất
            best, best_cnt = i, c
    return best

n, edges, t, want = examples[0]
print("sai:", wrong_tie(n, edges, t), "| đúng:", Solution().findTheCity(n, edges, t))

sai: 0 | đúng: 3


**Vì sao sai:** thành phố 0 và 3 cùng 2 láng giềng; `<` giữ thành phố gặp trước (0). Đề yêu cầu id **lớn nhất**. **Sửa:** `<=` khi duyệt tăng dần (hoặc duyệt giảm dần với `<`).

### Lỗi 2 — Đặt vòng `k` ở trong cùng (Floyd sai thứ tự)

In [22]:
def floyd_wrong_order(n, edges):
    INF = float("inf")
    d = [[INF] * n for _ in range(n)]
    for i in range(n):
        d[i][i] = 0
    for u, v, w in edges:
        d[u][v] = d[v][u] = w
    for i in range(n):
        for j in range(n):
            for k in range(n):              # SAI: k phải ở ngoài cùng
                d[i][j] = min(d[i][j], d[i][k] + d[k][j])
    return d

rng = random.Random(2)
found = None
for _ in range(2000):
    nn = rng.randint(4, 6)
    ee = random_graph(rng, nn, 0.5, 9)
    if floyd_wrong_order(nn, ee) != floyd(nn, ee):
        found = (nn, ee)
        break
nn, ee = found
bad, good = floyd_wrong_order(nn, ee), floyd(nn, ee)
diff = [(i, j, bad[i][j], good[i][j]) for i in range(nn) for j in range(nn) if bad[i][j] != good[i][j]]
print("phản ví dụ: n =", nn, "edges =", ee)
print("các ô sai (i, j, sai, đúng):", diff[:4])

phản ví dụ: n = 6 edges = [[1, 3, 4], [0, 4, 8], [0, 5, 9], [3, 5, 5]]
các ô sai (i, j, sai, đúng): [(0, 1, inf, 18), (1, 0, inf, 18), (1, 4, inf, 26), (4, 1, inf, 26)]


**Vì sao sai:** khi tính `d[i][j]` với vòng `k` bên trong, các ô `d[i][k]`, `d[k][j]` có thể **chưa** được cập nhật về giá trị ngắn nhất (chúng được tính ở các lượt `(i, j)` sau) ⇒ bỏ sót đường đi cần nhiều đỉnh trung gian. Ta tự động tìm một phản ví dụ ở trên. **Sửa:** `for k: for i: for j:` — đúng định nghĩa DP "cho phép thêm dần đỉnh trung gian".

### Lỗi 3 — Quên cạnh hai chiều

In [23]:
def wrong_directed(n, edges, distanceThreshold):
    INF = float("inf")
    d = [[INF] * n for _ in range(n)]
    for i in range(n):
        d[i][i] = 0
    for u, v, w in edges:
        d[u][v] = w                          # SAI: thiếu d[v][u]
    for k in range(n):
        for i in range(n):
            for j in range(n):
                d[i][j] = min(d[i][j], d[i][k] + d[k][j])
    return pick_city(n, d, distanceThreshold)

for n, edges, t, want in examples:
    print("sai:", wrong_directed(n, edges, t), "| đúng:", Solution().findTheCity(n, edges, t))

sai: 3 | đúng: 3
sai: 4 | đúng: 0


**Vì sao sai:** đề cho `from < to` nên chỉ thêm một chiều thì thành phố có id lớn **không đi ngược** được ⇒ thành phố `n−1` luôn có 0 láng giềng và bị chọn sai. **Sửa:** `d[u][v] = d[v][u] = w`.

### Lỗi 4 — Chọn `INF` quá nhỏ

In [24]:
def wrong_small_inf(n, edges, distanceThreshold):
    INF = 10**4                              # SAI: bằng ngưỡng tối đa
    d = [[INF] * n for _ in range(n)]
    for i in range(n):
        d[i][i] = 0
    for u, v, w in edges:
        d[u][v] = d[v][u] = w
    for k in range(n):
        for i in range(n):
            for j in range(n):
                d[i][j] = min(d[i][j], d[i][k] + d[k][j])
    return pick_city(n, d, distanceThreshold)

case = (4, [[1, 2, 1], [2, 3, 1]], 10**4)        # thành phố 0 cô lập
print("sai:", wrong_small_inf(*case), "| đúng:", Solution().findTheCity(*case))

sai: 3 | đúng: 0


**Vì sao sai:** thành phố 0 cô lập, khoảng cách từ nó tới mọi nơi là "vô cực" — nhưng `INF = 10^4 <= threshold = 10^4` nên bị đếm là đến được. Mọi thành phố đều tưởng có 3 láng giềng ⇒ code sai trả 3; đáp án đúng là 0 (0 láng giềng). **Sửa:** `INF` phải lớn hơn mọi giá trị có thể so sánh (`float('inf')` hoặc `10^9`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đến được với tổng độ dài ≤ X" ⇒ **khoảng cách ngắn nhất**, không phải BFS số cạnh.
- `n ≤ 100–400` và cần **mọi cặp** ⇒ **Floyd–Warshall** (3 vòng lặp, `k` ngoài cùng).
- Đồ thị thưa / `n` lớn / trọng số không âm ⇒ **Dijkstra** từ mỗi nguồn; có ngưỡng ⇒ cắt sớm.
- Có trọng số âm ⇒ Bellman–Ford (hoặc Floyd, nếu không có chu trình âm).
- Quy tắc hoà "id lớn nhất" ⇒ `<=` khi duyệt tăng; "id nhỏ nhất" ⇒ `<`.
- Luôn kiểm tra đồ thị **vô hướng** ⇒ thêm hai chiều.

## 15. Biến thể của bài

1. **Nhiều truy vấn ngưỡng** (Bài tập 5): Floyd một lần, sắp mỗi hàng, trả lời bằng `bisect`.
2. **LeetCode 743 — Network Delay Time:** Dijkstra một nguồn, lấy max khoảng cách.
3. **LeetCode 1462 — Course Schedule IV:** bao đóng bắc cầu bằng Floyd với `or`/`and`.
4. **LeetCode 2976 — Minimum Cost to Convert String I:** Floyd trên 26 ký tự.
5. Hoà thì chọn **id nhỏ nhất**, hoặc tìm thành phố có **nhiều** láng giềng nhất — chỉ đổi điều kiện so sánh.

In [25]:
def transitive_closure(n, prerequisites):
    """LeetCode 1462 (ý tưởng): reach[i][j] = có đường từ i đến j."""
    reach = [[False] * n for _ in range(n)]
    for a, b in prerequisites:
        reach[a][b] = True
    for k in range(n):
        for i in range(n):
            if reach[i][k]:
                for j in range(n):
                    if reach[k][j]:
                        reach[i][j] = True
    return reach

r = transitive_closure(4, [[0, 1], [1, 2], [2, 3]])
print("0 -> 3?", r[0][3], "| 3 -> 0?", r[3][0])

0 -> 3? True | 3 -> 0? False


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS đường đơn | liệt kê mọi đường, cắt theo ngưỡng | mũ | `O(n)` | kiểm chứng `n ≤ 7` |
| 2. Floyd–Warshall | DP thêm dần đỉnh trung gian | `O(n³)` | `O(n²)` | `n` nhỏ, cần mọi cặp, code ngắn |
| 3. Dijkstra × n | heap, dừng khi vượt ngưỡng | `O(n(n+E)log n)` | `O(n+E)` | đồ thị thưa, ngưỡng nhỏ |
| 4. Bellman–Ford × n | nới lỏng mọi cạnh `n−1` lượt | `O(n²E)` | `O(n+E)` | trọng số âm, mục đích học |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Với đồ thị Ví dụ 1 nhưng `distanceThreshold = 2`, đáp án là gì?

### Bài 2 (tính tay)
`n = 4`, `edges = [[0,1,1],[1,2,2],[0,2,4],[2,3,1]]`, `distanceThreshold = 3`. Tính ma trận khoảng cách, số láng giềng mỗi thành phố và đáp án.

### Bài 3 (điền TODO)
Hoàn thiện `floyd_todo`.

### Bài 4 (tự viết lại)
Viết lại Dijkstra với **mảng** `dist` và tự đếm, đặt tên `find_city_ex4`.

### Bài 5 (mini challenge)
Viết `answer_queries(n, edges, thresholds)` trả đáp án cho **từng** ngưỡng trong danh sách (có thể tới `10^4` ngưỡng). Chạy Floyd **một lần**, rồi mỗi truy vấn `O(n log n)` bằng `bisect`.

In [26]:
# Bài 3, 4, 5 — khung (chỉ định nghĩa, không gọi)
def floyd_todo(n, edges):
    INF = float("inf")
    dist = [[INF] * n for _ in range(n)]
    # TODO: đường chéo = 0, cạnh hai chiều
    # TODO: ba vòng lặp k, i, j
    return dist

def find_city_ex4_todo(n, edges, distanceThreshold):
    # TODO
    pass

def answer_queries_todo(n, edges, thresholds):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Dùng ma trận ở mục 4. (2) Đếm ô `≤ 2`: 0 → 0, 1 → 2 (2, 3), 2 → 2, 3 → 2. (3) Nhỏ nhất là 0 láng giềng ở thành phố 0.

**Bài 2:** (1) `0→2` qua 1 dài 3 < 4. (2) `0→3 = 4`, `1→3 = 3`. (3) Số láng giềng `[2, 3, 3, 2]` ⇒ hoà 0 và 3 ⇒ 3.

**Bài 3:** (1) `dist[i][i] = 0`. (2) `dist[u][v] = dist[v][u] = w`. (3) `for k: for i: for j: dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])`.

**Bài 4:** (1) `dist = [inf] * n; dist[s] = 0`. (2) Bỏ qua bản ghi cũ `d > dist[u]`. (3) Đếm `d <= threshold` trừ 1.

**Bài 5:** (1) Floyd → mỗi hàng sắp tăng `sorted_rows[i]`. (2) Số láng giềng của `i` với ngưỡng `T` = `bisect_right(sorted_rows[i], T) − 1`. (3) Chọn theo quy tắc hoà.

## 19. Đáp án tham khảo

In [27]:
from bisect import bisect_right

# Bài 1, 2
b1 = Solution().findTheCity(4, examples[0][1], 2)
e2 = [[0, 1, 1], [1, 2, 2], [0, 2, 4], [2, 3, 1]]
d2 = floyd(4, e2)
cnt2 = [sum(1 for j in range(4) if j != i and d2[i][j] <= 3) for i in range(4)]
b2 = Solution().findTheCity(4, e2, 3)
print("Bài 1:", b1, "| Bài 2: dist =", d2, "số láng giềng =", cnt2, "đáp án =", b2)
assert b1 == 0 and cnt2 == [2, 3, 3, 2] and b2 == 3

# Bài 3
def floyd_sol(n, edges):
    INF = float("inf")
    dist = [[INF] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
    for u, v, w in edges:
        dist[u][v] = dist[v][u] = w
    for k in range(n):
        for i in range(n):
            for j in range(n):
                dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])
    return dist

# Bài 4
def find_city_ex4(n, edges, distanceThreshold):
    graph = build_graph(n, edges)
    best, best_cnt = -1, n + 1
    for s in range(n):
        dist = [float("inf")] * n
        dist[s] = 0
        heap = [(0, s)]
        while heap:
            d, u = heapq.heappop(heap)
            if d > dist[u]:
                continue
            for v, w in graph[u]:
                if d + w < dist[v]:
                    dist[v] = d + w
                    heapq.heappush(heap, (dist[v], v))
        c = sum(1 for d in dist if d <= distanceThreshold) - 1
        if c <= best_cnt:
            best, best_cnt = s, c
    return best

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(2, 15)
    edges = random_graph(rng, n, 0.4, 20)
    t = rng.randint(1, 40)
    assert floyd_sol(n, edges) == floyd(n, edges)
    assert find_city_ex4(n, edges, t) == Solution().findTheCity(n, edges, t)
print("Bài 1–4: qua")

Bài 1: 0 | Bài 2: dist = [[0, 1, 3, 4], [1, 0, 2, 3], [3, 2, 0, 1], [4, 3, 1, 0]] số láng giềng = [2, 3, 3, 2] đáp án = 3


Bài 1–4: qua


In [28]:
# Bài 5
def answer_queries(n, edges, thresholds):
    dist = floyd(n, edges)
    rows = [sorted(row) for row in dist]
    res = []
    for T in thresholds:
        best, best_cnt = -1, n + 1
        for i in range(n):
            c = bisect_right(rows[i], T) - 1
            if c <= best_cnt:
                best, best_cnt = i, c
        res.append(best)
    return res

print("Bài 5:", answer_queries(*examples[0][:2], [1, 2, 3, 4, 5]))
rng = random.Random(5)
for _ in range(100):
    n = rng.randint(2, 20)
    edges = random_graph(rng, n, 0.3, 30)
    qs = [rng.randint(1, 80) for _ in range(10)]
    assert answer_queries(n, edges, qs) == [Solution().findTheCity(n, edges, T) for T in qs]
print("Bài 5: 100 đồ thị × 10 truy vấn khớp lời giải từng truy vấn")

Bài 5: [0, 0, 0, 3, 3]


Bài 5: 100 đồ thị × 10 truy vấn khớp lời giải từng truy vấn


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template Floyd–Warshall:**

```python
INF = float('inf')
d = [[INF] * n for _ in range(n)]
for i in range(n): d[i][i] = 0
for u, v, w in edges: d[u][v] = d[v][u] = w     # vô hướng
for k in range(n):                               # k NGOÀI CÙNG
    for i in range(n):
        for j in range(n):
            if d[i][k] + d[k][j] < d[i][j]:
                d[i][j] = d[i][k] + d[k][j]
```

**Template chọn với hoà id lớn nhất:** duyệt `i` tăng, `if cnt <= best_cnt: best = i`.

**Checklist trước khi submit:**
- [ ] Cạnh hai chiều.
- [ ] `INF` đủ lớn (không bằng ngưỡng).
- [ ] Vòng `k` ngoài cùng.
- [ ] Không đếm chính thành phố (hoặc trừ 1).
- [ ] So sánh `<= distanceThreshold` (bao gồm biên).
- [ ] Hoà ⇒ id lớn nhất (`<=` khi duyệt tăng).
- [ ] Đúng chữ ký: `findTheCity(self, n, edges, distanceThreshold) -> int`.